# Stage1 外部领导性：SMH 发行人历史价格来源预检

原 Stage1 0bp 为最终父本，本卡仅检验 VanEck 官方 SMH 历史文件的日期、字段和 NAS 备份；冻结方案见 `plan.json`，首次 Cookie 跳转失败与机械修复保留。

In [1]:
import os,sys,json,hashlib,re,zipfile,tarfile,tempfile,subprocess,shlex,datetime,collections,math
from pathlib import Path
from xml.etree import ElementTree as ET
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_smh_soxx_leadership_source_20261023';plan=json.loads((card/'plan.json').read_text());repair=json.loads((card/'repair_plan.json').read_text());state=json.loads((card/'run_state.json').read_text());assert plan['frozen'] and repair['frozen'] and state['HTTP_GET_attempts']==2
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=root/'data/snapshots/stage1_smh_soxx_leadership_source_20261023.xlsx';headers=root/'data/snapshots/stage1_smh_soxx_leadership_source_20261023_repair.headers.txt';assert raw.is_file() and zipfile.is_zipfile(raw)
h= headers.read_text(errors='replace');statuses=[v.strip() for v in re.findall(r'^HTTP/[^\r\n]+',h,re.M)];content_types=re.findall(r'^content-type:\s*([^\r\n]+)',h,re.M|re.I)
assert statuses==['HTTP/2 302','HTTP/2 302','HTTP/2 200'] and content_types[-1]=='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
ns={'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
with zipfile.ZipFile(raw) as z:
 workbook=ET.fromstring(z.read('xl/workbook.xml'));sheet_names=[x.attrib['name'] for x in workbook.findall('.//m:sheet',ns)];assert len(sheet_names)==1
 shared=ET.fromstring(z.read('xl/sharedStrings.xml'));strings=[''.join(t.text or '' for t in si.findall('.//m:t',ns)) for si in shared.findall('m:si',ns)]
 sheet=ET.fromstring(z.read('xl/worksheets/sheet1.xml'));rows=sheet.findall('.//m:sheetData/m:row',ns)
 def cells(row):
  result={}
  for cell in row.findall('m:c',ns):
   ref=cell.attrib['r'];col=re.match(r'[A-Z]+',ref).group();value=cell.findtext('m:v',default='',namespaces=ns)
   result[col]=strings[int(value)] if cell.attrib.get('t')=='s' and value else value
  return result
 assert cells(rows[1])==dict(zip('ABCDEFGHIJ',['Date','NAV','Change','% Change','Last Trade','Volume','Premium/Discount','% Premium/Discount','AUM','Index Level']))
 observations=[];bad_date=[]
 for row in rows[2:]:
  v=cells(row);d=v.get('A','')
  try:day=datetime.datetime.strptime(d,'%m/%d/%Y').date()
  except ValueError:bad_date.append(d);continue
  def num(k):
   try:return float(v.get(k,'').replace(',',''))
   except ValueError:return math.nan
  observations.append({'date':day.isoformat(),'NAV':num('B'),'market_last_trade':num('E'),'volume':num('F'),'premium':num('G'),'premium_pct':num('H'),'AUM':num('I'),'index_level':num('J')})
assert len(observations)>3000 and len({x['date'] for x in observations})==len(observations)
obs=sorted(observations,key=lambda x:x['date']);years={}
for year in range(2011,2027):
 z=[x for x in obs if x['date'].startswith(str(year))];valid=[x for x in z if x['NAV']>0 and x['market_last_trade']>0 and math.isfinite(x['NAV']) and math.isfinite(x['market_last_trade'])]
 years[str(year)]={'dated_rows':len(z),'positive_NAV_and_market_price':len(valid),'first':z[0]['date'] if z else None,'last':z[-1]['date'] if z else None}
missing_by_col={k:sum(not math.isfinite(x[k]) for x in obs) for k in ('NAV','market_last_trade','volume','premium','premium_pct','AUM','index_level')}
weekend=sum(datetime.date.fromisoformat(x['date']).weekday()>=5 for x in obs)
transport={'attempt1':json.loads((card/'transport_attempt1.json').read_text()),'repair_plan_sha256':sha(card/'repair_plan.json'),'repair_GET_statuses':statuses,'repair_content_type':content_types[-1],'repair_headers_sha256':sha(headers),'raw_bytes':raw.stat().st_size,'raw_sha256':sha(raw),'redirects':len(statuses)-1,'cookie_values_recorded':False}
(card/'transport_receipt.json').write_text(json.dumps(transport,indent=2)+'\n')
report={'decision':'issuer_SMH_historical_NAV_market_price_source_candidate_timing_unqualified','plan_sha256':sha(card/'plan.json'),'repair_plan_sha256':sha(card/'repair_plan.json'),'raw_sha256':sha(raw),'raw_bytes':raw.stat().st_size,'sheet_names':sheet_names,'sheet_rows':len(rows),'dated_rows':len(obs),'unparsed_rows':len(bad_date),'first_date':obs[0]['date'],'last_date':obs[-1]['date'],'duplicate_dates':0,'weekend_rows':weekend,'missing_by_column':missing_by_col,'years':years,'2012_2022_positive_NAV_and_market_price':sum(years[str(y)]['positive_NAV_and_market_price'] for y in range(2012,2023)),'2023_2025_positive_NAV_and_market_price':sum(years[str(y)]['positive_NAV_and_market_price'] for y in range(2023,2026)),'first_publication_clock_qualified':False,'split_adjustment_policy_qualified':False,'SOXX_or_SMH_future_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(report,indent=2)+'\n')
settings=_settings(root);members=[raw,card/'plan.json',card/'repair_plan.json',card/'transport_attempt1.json',card/'transport_receipt.json',card/'source_audit.json']
with tempfile.TemporaryDirectory(prefix='stage1-smh-source-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for p in members:out.add(p,arcname=str(p.relative_to(root)))
 digest=sha(archive);response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research');receipt=response['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;restored=Path(tmp)/'restored.tar.gz'
 with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(p.relative_to(root)) for p in members}
  for p in members:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
archive_receipt={'sha256':digest,'size_bytes':receipt['size_bytes'],'nas_path':remote,'members':{str(p.relative_to(root)):sha(p) for p in members},'remote_and_scoped_restore_verified':True};(card/'nas_source_archive.json').write_text(json.dumps(archive_receipt,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'source_audit':'source_audit.json','source_archive':'nas_source_archive.json','source_NAS_sha256':digest,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Current retrospective SMH issuer workbook does not prove historical first-public delivery by Stage1 09:46.','NAV is an after-close value and market Last Trade is not an open executable quote.','Fund begins December 2011; earliest Stage1 training period cannot have SMH history.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='source_only_finished';state['raw_sha256']=sha(raw);state['nas_sha256']=digest;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(json.dumps({'dated_rows':len(obs),'first':obs[0]['date'],'last':obs[-1]['date'],'years':years,'missing':missing_by_col,'NAS':digest},indent=2))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "dated_rows": 4090,
  "first": "2011-12-20",
  "last": "2026-10-21",
  "years": {
    "2011": {
      "dated_rows": 8,
      "positive_NAV_and_market_price": 8,
      "first": "2011-12-20",
      "last": "2011-12-30"
    },
    "2012": {
      "dated_rows": 250,
      "positive_NAV_and_market_price": 250,
      "first": "2012-01-03",
      "last": "2012-12-31"
    },
    "2013": {
      "dated_rows": 252,
      "positive_NAV_and_market_price": 252,
      "first": "2013-01-02",
      "last": "2013-12-31"
    },
    "2014": {
      "dated_rows": 252,
      "positive_NAV_and_market_price": 252,
      "first": "2014-01-02",
      "last": "2014-12-31"
    },
    "2015": {
      "dated_rows": 252,
      "positive_NAV_and_market_price": 252,
      "first": "2015-01-02",
      "last": "2015-12-31"
    },
    "2016": {
      "dated_rows": 252,
      "positive_NAV_and_market_price": 252,
      "first": "2016-01-04",
      "last": "2016-12-30"
    },
    "2017": {
      "dated_rows": 251,
   

# SMH 相对 SOXX 领导性：发行人原始历史源预检

VanEck 官方 [SMH 页面](https://www.vaneck.com/us/en/investments/semiconductor-etf-smh/overview)的“[NAV & Premium/Discount History](https://www.vaneck.com/us/en/investments/semiconductor-etf-smh/downloads/fundhistoprices/)”下载，首次 GET 因无 Cookie 进入 302 循环；冻结仅启用 Cookie 保存的机械修复后，第二次 GET 返回 XLSX。失败头哈希、修复方案和无 Cookie 值的传输收据均保留。原始文件 SHA-256 `c8b0c443…`，Yanxuan NAS 归档 SHA-256 `11c83f7f…` 已远端哈希及限定恢复。

文件一张表、4,090 个唯一日期，2011-12-20 至 2026-10-21。2012–2022 有 **2,768** 行正值 NAV 与 Last Trade，2023–2025 有 **1,020** 行（含周末重复值）；全表 NAV、Last Trade、溢折价均无缺值，Volume 缺 361 行。SMH 2011 年 12 月才开始，原 Stage1 2011 年大部分训练日没有该来源。VanEck 明确 NAV 于营业日收盘后计算；因此在 Stage1 09:46 决策时，只可能使用已经公布的前一交易日数据。文件是当前回溯下载，历史首次送达时刻、修订与拆分调整仍需验证。[VanEck 2023 年拆分公告](https://www.vaneck.com/us/en/press-releases/smh-vaneck-semiconductor-etf-share-split-press-release.pdf)记载 2023-05-05 开盘起 2 比 1 拆分，当前文件跨拆分日价格连续，提示回溯调整；不能将原始价位当作当时发行值。

**决定：来源候选，尚不构成 PIT 特征。** 这个 SMH 大盘龙头相对 SOXX 的价格领导性机制不同于已失败的 XSD 等权广度与 SOXX 自身趋势，但“不同”不证明有增量。当前零 SOXX/SMH 未来结果读取、零拟合、零账户。下一步先以原生 Qlib 日历核对完整交易日、拆分与前日可用时点，再冻结与原 Stage1 39 项的匹配信息门；任何通过都须在完整自有状态 0bp Tiered 账户、2/5bp 和独立前向证据上重新验证。
